In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("HW_07.ipynb")

In [ ]:
rng_seed = 42

## Homework 7

## <em> Linear Regression, Regularization, and Logistic & Softmax Regression</em>
<br>
This notebook is arranged in cells. Texts are usually written in the markdown cells, and here you can use html tags (make it bold, italic, colored, etc). You can double click on this cell to see the formatting.<br>
<br>
The ellipsis (...) are provided where you are expected to write your solution but feel free to change the template (not over much) in case this style is not to your taste. <br>
<br>
<em>Hit "Shift-Enter" on a code cell to evaluate it.  Double click a Markdown cell to edit. </em><br>

### Imports

In [ ]:
import numpy as np
from scipy.integrate import quad
#For plotting
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

#### Problem 1 - Ising Model

In earlier HW, we did a simple ML analysis by fitting datasets generated by polynomials in the presence of noise, and this highlighted the fundamental tension common to all ML models between how well we fit the training dataset and predictions on new data.

Here, we consider the problem of learning the Hamiltonian for the Ising model (https://en.wikipedia.org/wiki/Ising_model) using the linear regression. This is a lattice model proposed to explain ferromagnetism in materials. In other physics courses, you learned that elementary particles have an intrinsic property called spin, which carries magnetic moments. The magnetism of a bulk material is made up of the magnetic dipole moments of the atomic spins inside the material. The classical Ising model postulates a lattice with a spin $S$ on each site.

Now consider the 1D Ising model with nearest-neighbor interactions

$$H[\boldsymbol{S}]=-J\sum_{j=1}^L S_{j}S_{j+1}$$

on a chain of length $L$ with periodic boundary conditions and $S_j=\pm 1$ Ising spin variables. $J$ is the nearest-neighbor spin interaction

With $J=1$, we draw a large number of spin configurations. We can draw them $n$ number of times: we have $n$ number of $\boldsymbol{S}^i$, which is a vector of length $L$. Hence, $\boldsymbol{S}$ is a matrix of $n \times L$.



<!-- BEGIN QUESTION -->

<span style="color:blue"> <i> 1. You are given 1000 random Ising states with $L=40$. (i.e. this state matrix $\boldsymbol{S}$ should have the dimension $1000 \times 40$, and its array elements are either 1 or -1.) Define a function which computes the energies $H$ given $\boldsymbol{S}$. Calculate the energies of the first 10 states. (Do not shuffle the states!) </i></span> <br>

Hint: Each state $\boldsymbol{S}^i$ has its own energy, so $H[\boldsymbol{S}]$ is a vector of length $n=1000$.

We adopt the periodic boundary conditions, so when $j=L$, $j+1=1$.

In [ ]:
S = np.loadtxt("./state.txt")
print( np.shape(S) )
print( S )

In [ ]:
L = 40

def ising_energies(S, L):
    """
    Calculate the Ising energies for states S.
    H = -J * sum_{j=1}^L S_j * S_{j+1}
    With J=1 and periodic boundary conditions.
    """
    # For each state, compute the sum of S_j * S_{j+1}
    # Use periodic boundary: S[:, -1] wraps to S[:, 0]
    energies = -np.sum(S[:, :-1] * S[:, 1:], axis=1) - S[:, -1] * S[:, 0]
    return energies

# calculate Ising energies
energies = ising_energies(S, L)

print( energies[0:10] )

In [ ]:
grader.check("q1.1")

<!-- END QUESTION -->

Now, suppose you do not have the knowledge of the above Hamiltonian. Instead, you are given a data set of $i=1\ldots n$ points of the form $\{(H[\boldsymbol{S}^i],\boldsymbol{S}^i)\}$. Your task is to learn the Hamiltonian using Linear regression techniques.

In the absence of any prior knowledge, one sensible choice is the all-to-all Ising model

$$
H_\mathrm{model}[\boldsymbol{S}^i] = - \sum_{j=1}^L \sum_{k=1}^L J_{j,k}S_{j}^iS_{k}^i.
$$
Notice that this model is uniquely defined by the non-local coupling strengths $J_{jk}$ which we want to learn. Importantly, this model is linear in ${\mathbf J}$ which makes it possible to use linear regression.

To apply linear regression, we would like to recast this model in the form
$$
H_\mathrm{model}^i \equiv \mathbf{X}^i \cdot \mathbf{J},
$$

where the vectors $\mathbf{X}^i$ represent all two-body interactions $\{S_{j}^iS_{k}^i \}_{j,k=1}^L$, and the index $i$ runs over the samples in the data set. To make the analogy complete, we can also represent the dot product by a single index $p = \{j,k\}$, i.e. $\mathbf{X}^i \cdot \mathbf{J}=X^i_pJ_p$. Note that the regression model does not include the minus sign, so we expect to learn negative $J$'s.

<!-- BEGIN QUESTION -->

<span style="color:blue"> <i> 2. Create the matrix $\mathbf{X}$. Print $\mathbf{X}$. </i></span> <br>

Hint: For each state $i$, we have the state vector $\boldsymbol{S}^i$. $\mathbf{X}^i$ = $\boldsymbol{S}^i_{.T} \otimes \boldsymbol{S}^i_{.T}$, where $\otimes$ is the outer product. (https://en.wikipedia.org/wiki/Outer_product)

The dimension of $\mathbf{X}^i$ is $L \times L$. Hence, $\mathbf{X}$ has the diemension $n \times L \times L$. Reshape it so that it has the dimension $n \times L*L$ ($1000 \times 1600$).

You can either use the for-loop or use np.einsum to do the outer product (https://docs.scipy.org/doc/numpy-1.15.1/reference/generated/numpy.einsum.html).

In [ ]:
# Method 1: Using einsum (more efficient)
X = np.einsum('ij,ik->ijk', S, S)
X = X.reshape(S.shape[0], -1)  # Reshape to (n, L*L)

# Alternative Method 2: Using loop
# X = np.zeros((S.shape[0], L, L))
# for i in range(S.shape[0]):
#     X[i] = np.outer(S[i], S[i])
# X = X.reshape(S.shape[0], -1)

print(f"Shape of X: {X.shape}")
print(X)

In [ ]:
# Print first few rows to verify
print(f"First row of X:\n{X[0][:20]}")

In [ ]:
grader.check("q1.2")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

We can now do the linear regression.
$$
H_\mathrm{model}^i \equiv \mathbf{X}^i \cdot \mathbf{J},
$$
Hence, you have data ($\mathbf{X}, H$)

<span style="color:blue"> <i> 3. Split the data into training and test samples. We choose that the first 70% of $n$ states are training samples, the remaining 30% test samples. No need to shuffle the data because we are already given the random set of states. Print the diemension of training and test samples.</i></span> <br>

Hint: Here, H means $H[\boldsymbol{S}]$ or $H[\mathbf{X}]$ we calculated in Part 1.

In [ ]:
# Split into 70% training and 30% test
n_samples = X.shape[0]
split_idx = int(0.7 * n_samples)

X_train = X[:split_idx]
Y_train = energies[:split_idx]
X_test = X[split_idx:]
Y_test = energies[split_idx:]

print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

In [ ]:
grader.check("q1.3")

<!-- END QUESTION -->

In earlier HW, you used "linear_model.LinearRegression()" from scikit-learn to do the linear regression and found that using a complex model can result in overfitting. To resolve such issues, we use regularization in machine learning. A regression model that uses $L_1$ regularization technique is called Lasso Regression (https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) and model which uses $L_2$ is called Ridge Regression (https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html).

First, set up Lasso and Ridge regression models.

In [ ]:
from sklearn import linear_model
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
%matplotlib inline

ridge = linear_model.Ridge()
lasso = linear_model.Lasso()

<!-- BEGIN QUESTION -->

<span style="color:blue"> <i> 4. Compute the MSE (mean squared error) on train and test sets for Ridge Regression and Lasso Regression. (Use lambda values of np.logspace(-4,4,10). For better results, you can increase the numbers of lambdas, i.e. replace 10 by 20 or use linspace instead, but this will take longer time to compute.) Make a plot to show MSE as a function of regularization strength. (https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html) </i></span> <br>

In [ ]:
from sklearn.metrics import mean_squared_error

# Define lambda values
lmbdas = np.logspace(-4, 4, 10)

# Initialize arrays to store errors
train_errors_ridge = np.zeros(len(lmbdas))
test_errors_ridge = np.zeros(len(lmbdas))
train_errors_lasso = np.zeros(len(lmbdas))
test_errors_lasso = np.zeros(len(lmbdas))

# Loop over lambda values
for i, lmbda in enumerate(lmbdas):
    # Ridge Regression
    ridge_model = linear_model.Ridge(alpha=lmbda)
    ridge_model.fit(X_train, Y_train)
    
    Y_train_pred_ridge = ridge_model.predict(X_train)
    Y_test_pred_ridge = ridge_model.predict(X_test)
    
    train_errors_ridge[i] = mean_squared_error(Y_train, Y_train_pred_ridge)
    test_errors_ridge[i] = mean_squared_error(Y_test, Y_test_pred_ridge)
    
    # Lasso Regression
    lasso_model = linear_model.Lasso(alpha=lmbda, max_iter=10000)
    lasso_model.fit(X_train, Y_train)
    
    Y_train_pred_lasso = lasso_model.predict(X_train)
    Y_test_pred_lasso = lasso_model.predict(X_test)
    
    train_errors_lasso[i] = mean_squared_error(Y_train, Y_train_pred_lasso)
    test_errors_lasso[i] = mean_squared_error(Y_test, Y_test_pred_lasso)

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Ridge Regression
ax1.semilogx(lmbdas, train_errors_ridge, 'b-o', label='Train')
ax1.semilogx(lmbdas, test_errors_ridge, 'r-s', label='Test')
ax1.set_xlabel('Regularization strength (lambda)', fontsize=12)
ax1.set_ylabel('MSE', fontsize=12)
ax1.set_title('Ridge Regression', fontsize=14)
ax1.legend()
ax1.grid(True)

# Lasso Regression
ax2.semilogx(lmbdas, train_errors_lasso, 'b-o', label='Train')
ax2.semilogx(lmbdas, test_errors_lasso, 'r-s', label='Test')
ax2.set_xlabel('Regularization strength (lambda)', fontsize=12)
ax2.set_ylabel('MSE', fontsize=12)
ax2.set_title('Lasso Regression', fontsize=14)
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
grader.check("q1.4")

<!-- END QUESTION -->

***

#### Problem 1 continued.

Rather than distinguishing between ordered and disordered phases, we now train a logistic regression model to distinguish different phases of matter in Ising model at finite temperature.

Load data from different phases. (ordered, critical and disordered)

In [ ]:
import pickle,os
from sklearn.model_selection import train_test_split

# load data

# state vector
file_name = "./Ising2DFM_reSample_L40_T=All_labels.pkl"
state_vector = pickle.load(open(file_name,'rb'))

# ordered phases
file_name = "./Ising2DFM_reSample_L40_T=1.00.pkl"
data = pickle.load(open(file_name,'rb'))
data = np.unpackbits(data).reshape(-1, 1600)
data=data.astype('int')
data[np.where(data==0)]=-1 # map 0 state to -1 (Ising variable can take values +/-1)

X_ordered=data
Y_ordered=state_vector[30000:40000]

# critical phases
file_name = "./Ising2DFM_reSample_L40_T=2.25.pkl"
data = pickle.load(open(file_name,'rb'))
data = np.unpackbits(data).reshape(-1, 1600)
data=data.astype('int')
data[np.where(data==0)]=-1

X_critical=data
Y_critical=state_vector[80000:90000]

# disordered phases
file_name = "./Ising2DFM_reSample_L40_T=3.00.pkl"
data = pickle.load(open(file_name,'rb'))
data = np.unpackbits(data).reshape(-1, 1600)
data=data.astype('int')
data[np.where(data==0)]=-1

X_disordered=data
Y_disordered=state_vector[110000:120000]

L = 40

You have $\textbf{X}$ for ordered, critical and disordered phases and corresponding state vector $Y$.
For each phase (ordered, critical or disordered), we have 500 different $40\times 40$ square lattices. So $\textbf{X}$ has the dimension $500 \times 40 \times 40$. We reshape it into $500 \times 40*40$ = $500 \times 1600$. The state vector is a vector of length 500.

Run the below cell to plot examples of typical states of the 2D Ising model for three different temperatures.

In [ ]:
# plot few Ising states
from mpl_toolkits.axes_grid1 import make_axes_locatable

cmap_args=dict(cmap='plasma_r')

fig, axarr = plt.subplots(nrows=1, ncols=3)

axarr[0].imshow(X_ordered[100].reshape(L,L),**cmap_args)
axarr[0].set_title('$\\mathrm{ordered\\ phase}$',fontsize=16)
axarr[0].tick_params(labelsize=16)

axarr[1].imshow(X_critical[100].reshape(L,L),**cmap_args)
axarr[1].set_title('$\\mathrm{critical\\ region}$',fontsize=16)
axarr[1].tick_params(labelsize=16)

im=axarr[2].imshow(X_disordered[100].reshape(L,L),**cmap_args)
axarr[2].set_title('$\\mathrm{disordered\\ phase}$',fontsize=16)
axarr[2].tick_params(labelsize=16)

fig.subplots_adjust(right=2.0)

plt.show()

<!-- BEGIN QUESTION -->

<span style="color:blue"> <i> 5. Combine ordered phase samples and disordered phase samples using np.concatenate. Using train_test_split (https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), split it into training and test samples. Set train_size = 0.5. (50% of $X$ is our training samples.) Print the dimension of training and test samples. </i></span> <br>

Using logistic regression, we will investigate how accurately we can distinguish between ordered and disordered phases.

In [ ]:
# Combine ordered and disordered phases
X = np.concatenate((X_ordered, X_disordered))
Y = np.concatenate((Y_ordered, Y_disordered))

# Split into training and test samples
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.5, random_state=rng_seed)

print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

In [ ]:
grader.check("q1.5")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

Here, we compare the performance of two different optimization routines: a liblinear (the default one for scikit's logistic regression, https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), and stochastic gradient descent (SGD, https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html). It is important to note that all these methods have built-in regularizers, and doing regularization is crucial in order to prevent overfitting.

For each optimization routine, do the following:

1. Choose the regularization parameter $\lambda$.
2. Define the logistic regressor <br>
    e.g. $\textbf{liblinear}$: `logreg=linear_model.LogisticRegression(C=1.0/lambda,random_state=1,verbose=0,max_iter=1E3,tol=1E-5)`<br>
    e.g. $\textbf{SGD}$: `logreg_SGD = linear_model.SGDClassifier(loss='log', penalty='l2', alpha=lmbda, max_iter=100, shuffle=True, random_state=1, learning_rate='optimal')`<br>
    Use the above parameters, but you can play with them if you wish.
3. Fit the model<br>
    e.g. `logreg.fit(training X samples, training H samples)`
4. Compute the mean accuracy on the given data.
    e.g. `logreg.score(training or test X samples, training or test H samples)`

<span style="color:blue"> <i> 6. Let `lambda = np.logspace(-5,5,11)`. Compute the mean accuracy for each lambda value and plot it as a function of lambda. Do both liblinear and SGD. Also, show results for both training, test samples, and critical phase samples. (6 plots) What do you find? </i></span> <br>


In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier

# Define lambda values
lmbdas = np.logspace(-5, 5, 11)

# Initialize arrays to store accuracies
train_acc_liblinear = np.zeros(len(lmbdas))
test_acc_liblinear = np.zeros(len(lmbdas))
critical_acc_liblinear = np.zeros(len(lmbdas))

train_acc_SGD = np.zeros(len(lmbdas))
test_acc_SGD = np.zeros(len(lmbdas))
critical_acc_SGD = np.zeros(len(lmbdas))

# Loop over lambda values
for i, lmbda in enumerate(lmbdas):
    # Liblinear
    logreg_lib = LogisticRegression(C=1.0/lmbda, solver='liblinear', 
                                    random_state=1, verbose=0, 
                                    max_iter=int(1e3), tol=1e-5)
    logreg_lib.fit(X_train, Y_train)
    
    train_acc_liblinear[i] = logreg_lib.score(X_train, Y_train)
    test_acc_liblinear[i] = logreg_lib.score(X_test, Y_test)
    critical_acc_liblinear[i] = logreg_lib.score(X_critical, Y_critical)
    
    # SGD
    logreg_SGD = SGDClassifier(loss='log_loss', penalty='l2', alpha=lmbda, 
                               max_iter=100, shuffle=True, random_state=1, 
                               learning_rate='optimal')
    logreg_SGD.fit(X_train, Y_train)
    
    train_acc_SGD[i] = logreg_SGD.score(X_train, Y_train)
    test_acc_SGD[i] = logreg_SGD.score(X_test, Y_test)
    critical_acc_SGD[i] = logreg_SGD.score(X_critical, Y_critical)

# Plotting
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Liblinear plots
axes[0, 0].semilogx(lmbdas, train_acc_liblinear, 'b-o')
axes[0, 0].set_xlabel('Lambda', fontsize=12)
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].set_title('Liblinear - Training', fontsize=14)
axes[0, 0].grid(True)

axes[0, 1].semilogx(lmbdas, test_acc_liblinear, 'r-s')
axes[0, 1].set_xlabel('Lambda', fontsize=12)
axes[0, 1].set_ylabel('Accuracy', fontsize=12)
axes[0, 1].set_title('Liblinear - Test', fontsize=14)
axes[0, 1].grid(True)

axes[0, 2].semilogx(lmbdas, critical_acc_liblinear, 'g-^')
axes[0, 2].set_xlabel('Lambda', fontsize=12)
axes[0, 2].set_ylabel('Accuracy', fontsize=12)
axes[0, 2].set_title('Liblinear - Critical', fontsize=14)
axes[0, 2].grid(True)

# SGD plots
axes[1, 0].semilogx(lmbdas, train_acc_SGD, 'b-o')
axes[1, 0].set_xlabel('Lambda', fontsize=12)
axes[1, 0].set_ylabel('Accuracy', fontsize=12)
axes[1, 0].set_title('SGD - Training', fontsize=14)
axes[1, 0].grid(True)

axes[1, 1].semilogx(lmbdas, test_acc_SGD, 'r-s')
axes[1, 1].set_xlabel('Lambda', fontsize=12)
axes[1, 1].set_ylabel('Accuracy', fontsize=12)
axes[1, 1].set_title('SGD - Test', fontsize=14)
axes[1, 1].grid(True)

axes[1, 2].semilogx(lmbdas, critical_acc_SGD, 'g-^')
axes[1, 2].set_xlabel('Lambda', fontsize=12)
axes[1, 2].set_ylabel('Accuracy', fontsize=12)
axes[1, 2].set_title('SGD - Critical', fontsize=14)
axes[1, 2].grid(True)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Both methods achieve high accuracy on training and test sets for ordered vs disordered classification")
print("- Critical phase samples show lower accuracy, as expected since they are intermediate states")
print("- Regularization strength affects overfitting: too low lambda can lead to overfitting, too high underfits")
print("- SGD and liblinear show similar performance trends")

<!-- END QUESTION -->

***

#### Problem 2 - Back to MNIST

Now, we generalize logistic regression to the case of multiple categories which is called Softmax regression. A paradigmatic example of SoftMax regression is the MNIST classification problem. The goal is to find a statistical model which recognizes the ten handwritten digits. There are numerous practical applications of such a task, pretty much anywhere one can imagine dealing with large quantities of numbers.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import check_random_state

# Load MNIST data
X = np.loadtxt("./mnistX.dat")
Y = np.loadtxt("./mnistY.dat")

In [ ]:
np.shape(X)

<!-- BEGIN QUESTION -->

"$X$" contains information about the given MNIST digits. We have a 28x28 pixel grid, so each image is a vector of length 784; we have 3800 images (digits), so $X$ is a 3800x784 matrix. "$Y$" is a label (0-9; the category to which each image belongs) vector of length 3800.

<span style="color:blue"> <i> 1. Randomly shuffle data and split them into training and test samples using train_test_split. Let train_size = 0.8. Print the dimension of training and test samples. </i></span> <br>


In [ ]:
random_state = check_random_state(rng_seed)

# Shuffle and split the data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.8, 
                                                    random_state=random_state, 
                                                    shuffle=True)

print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

In [ ]:
grader.check("q2.1")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<span style="color:blue"> <i> 2. Choose any five images and show what the images look like. Print the corresponding labels. </i></span> <br>

Hint: each image is a vector of length 784. So reshape it into a 28x28 matrix.<br>
$\ \ $ `X_0 = X_train[0]`<br>
$\ \ $  `X_0 = X_0.reshape((28, 28))`<br>
Then, make a plot using imshow.<br>
$\ \ $  `plt.imshow(X_0, cmap=plt.cm.gray)`

In [ ]:
# Choose 5 random images
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for i in range(5):
    # Get the image and reshape
    X_img = X_train[i].reshape((28, 28))
    
    # Plot
    axes[i].imshow(X_img, cmap=plt.cm.gray)
    axes[i].set_title(f'Label: {int(Y_train[i])}', fontsize=12)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print(f"Labels: {Y_train[:5].astype(int)}")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

Now, do logistic regression in the following way:

1. Scale data to have zero mean and unit variance (https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) <br>
  `scaler = StandardScaler()` <br>
  `X_train = scaler.fit_transform(X_train)` <br>
  `X_test = scaler.transform(X_test)` <br>
2. Make an instance of the model using LogisticRegression (https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). Try "liblinear" and "sag" optimization algorithms.<br>
  $\textbf{liblinear}$: Use solver='liblinear' and use L1 norm in the penalization (penalty='l1'). Also set C=1e5 and tol=.3<br>
  $\textbf{sag}$: Use solver='sag' and use L2 norm in the penalization (penalty='l2'). Also set C=1e5 and tol=.1<br>
  e.g. `model = LogisticRegression(...)` <br>
3. Train the model on the data <br>
  e.g. `model.fit(training X sample, training Y samples)` <br>
4. Predict the labels of test data.<br>
  e.g. `digit_predict = model.predict(test X samples)` <br>
5. Compute the accuracy<br>
  e.g. `model.score(test X samples, test Y samples)` <br>
  
<span style="color:blue"> <i> 3. Using both liblinear and sag solvers, compute the accuracy of the test samples. Also, measure the training time (how long it takes to train the model on the data) using time.time().  </i></span> <br>


In [ ]:
import time

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Liblinear solver
t0 = time.time()
liblinear = LogisticRegression(solver='liblinear', penalty='l1', C=1e5, tol=0.3, 
                               random_state=rng_seed, max_iter=1000)
liblinear.fit(X_train_scaled, Y_train)
run_time_liblinear = time.time() - t0
score_liblinear = liblinear.score(X_test_scaled, Y_test)

print('liblinear:')
print('Run time: %.3f s' % run_time_liblinear)
print('accuracy: %.3f' % score_liblinear)

# SAG solver
t0 = time.time()
sag = LogisticRegression(solver='sag', penalty='l2', C=1e5, tol=0.1, 
                        random_state=rng_seed, max_iter=1000)
sag.fit(X_train_scaled, Y_train)
run_time_SGD = time.time() - t0
score_SGD = sag.score(X_test_scaled, Y_test)

print('sag:')
print('Run time: %.3f s' % run_time_SGD)
print('accuracy: %.3f' % score_SGD)

In [ ]:
grader.check("q2.3")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<span style="color:blue"> <i> 4. Choose any 15 images and show what the images look like. What are the predicted labels corresponding to them? Take a look at the misclassified samples. Use the sag solver. </i></span> <br>


In [ ]:
# Predict on test set
Y_pred = sag.predict(X_test_scaled)

# Plot 15 images
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.ravel()

for i in range(15):
    # Get the image and reshape
    X_img = X_test[i].reshape((28, 28))
    
    # Determine if correctly classified
    true_label = int(Y_test[i])
    pred_label = int(Y_pred[i])
    is_correct = true_label == pred_label
    
    # Plot
    axes[i].imshow(X_img, cmap=plt.cm.gray)
    
    # Color code: green for correct, red for incorrect
    color = 'green' if is_correct else 'red'
    axes[i].set_title(f'True: {true_label}, Pred: {pred_label}', 
                     fontsize=10, color=color)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

# Find and display misclassified samples
misclassified_indices = np.where(Y_test != Y_pred)[0]
print(f"\nNumber of misclassified samples: {len(misclassified_indices)} out of {len(Y_test)}")
print(f"Misclassification rate: {len(misclassified_indices)/len(Y_test):.3f}")

if len(misclassified_indices) > 0:
    print(f"\nFirst few misclassified examples:")
    for idx in misclassified_indices[:5]:
        print(f"  Index {idx}: True label = {int(Y_test[idx])}, Predicted = {int(Y_pred[idx])}")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<span style="color:blue"> <i> 5. Obtain the coefficient of the features for the model using the sag solver. (`coef = sag.coef_`) This is a 10x784 matrix. (number of classes x number of features) Reshape it into 28x28 and make a plot for each class. How do they look? Can you recognize the digits? </i></span> <br>


In [ ]:
# Get coefficients
coef = sag.coef_
print(f"Coefficient shape: {coef.shape}")

# Plot coefficients for each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i in range(10):
    # Reshape to 28x28
    coef_img = coef[i].reshape((28, 28))
    
    # Plot
    im = axes[i].imshow(coef_img, cmap='seismic', interpolation='nearest')
    axes[i].set_title(f'Digit {i}', fontsize=14)
    axes[i].axis('off')
    
    # Add colorbar
    divider = make_axes_locatable(axes[i])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- The coefficient images show the 'template' that the model learned for each digit")
print("- Positive (red) regions indicate pixels that support that digit classification")
print("- Negative (blue) regions indicate pixels that oppose that digit classification")
print("- The patterns are recognizable and resemble the actual digit shapes")
print("- This visualization helps understand what features the model uses for classification")

<!-- END QUESTION -->

***

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Submit the notebook (not the zip) to Gradescope! For the PDF submission, use Cmd + P to get the notebook!

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)